# Phase 10 — Launch the full-system Streamlit UI on Colab

Wires Phases 5 (disease) + 6 (soil) + 7 (RAG) + 8 (integration) + 9 (explainability) into a single live demo.

Run the cells in order. The last cell starts streamlit + a localtunnel and prints a public URL you can open in any browser. URL is session-bound — Colab session timeout kills it.

**Requirements:** T4 GPU runtime, HF Write token (for the private corpus + Llama-3.1-8B).

In [ ]:
# Cell 2 — clean clone + deps + HF login + GPU check.
# GIT_LFS_SKIP_SMUDGE=1 dodges the LFS bandwidth stall on the .pt files.
import os, shutil, subprocess, sys

REPO_PATH = "/content/iks-rag-thesis"
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"

# Step OUT of REPO_PATH before deleting it — otherwise cwd becomes invalid and
# subsequent subprocess (git, pip) fail with a FileNotFoundError / exit-128 cascade.
os.chdir("/content")
shutil.rmtree(REPO_PATH, ignore_errors=True)
env = os.environ.copy()
env["GIT_LFS_SKIP_SMUDGE"] = "1"   # skip the LFS binaries

r = subprocess.run(["git", "clone", REPO_URL, REPO_PATH], env=env, capture_output=True, text=True)
if r.returncode != 0:
    print("git clone STDOUT:", r.stdout); print("git clone STDERR:", r.stderr)
    raise RuntimeError(f"git clone failed (exit {r.returncode})")

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)
print("Repo at:", os.getcwd())

# Phase 10 runtime deps for streamlit_app.py's full import chain.
# - PyPI package is "grad-cam" (import name pytorch_grad_cam).
# - rank_bm25 needed by HybridRetriever's sparse index.
DEPS = [
    "streamlit>=1.36",
    "timm>=1.0",
    "albumentations>=1.4",
    "rembg>=2.0",
    "onnxruntime>=1.18",
    "huggingface_hub>=0.24",
    "datasets>=2.20",
    "transformers>=4.44,<4.50",
    "accelerate>=0.34",
    "bitsandbytes>=0.44",
    "sentence-transformers>=3.0",
    "chromadb>=0.5",
    "grad-cam>=1.5",
    "rank_bm25>=0.2.2",
    "pyyaml>=6.0",
    "pydantic>=2.7",
    "opencv-python-headless",
    "matplotlib>=3.7",
]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], capture_output=True, text=True)
if r.returncode != 0:
    print("pip STDOUT (last 40):", "\n".join(r.stdout.splitlines()[-40:]))
    print("pip STDERR (last 40):", "\n".join(r.stderr.splitlines()[-40:]))
    raise RuntimeError(f"pip install failed (exit {r.returncode})")

# ultralytics (YOLO leaf cropper for the crop-first C-PD pipeline) in a SEPARATE
# pip call — bundling it with the pinned deps above can trip pip's resolver.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.0"], check=True)
print("deps installed")

from huggingface_hub import HfApi, login
login()
print("HF user:", HfApi().whoami().get("name"))

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM free {free/1024**3:.2f} / {total/1024**3:.2f} GiB")

## Cell 3 — start Streamlit + cloudflared tunnel, print the public URL

Streamlit runs on port 8501 in the background. `cloudflared` (Cloudflare's free quick-tunnel) exposes it to a public `https://*.trycloudflare.com` URL. No auth, no gate page, and — crucially — it handles Streamlit's lazy-loaded JS chunks correctly, which loca.lt does NOT. The session dies on Colab timeout (expected for a demo).

In [ ]:
import os, re, subprocess, time, urllib.request

# (a) install cloudflared (one-time per Colab session, ~15s)
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        ["wget", "-q",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", "/usr/local/bin/cloudflared"],
        check=True,
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
print("cloudflared installed")

# (b) start streamlit (background; logs to /content/streamlit.log)
# Streamlit subprocesses do NOT inherit the notebook's sys.path. Without
# PYTHONPATH the in-app `from app import config` raises ModuleNotFoundError.
REPO_PATH = "/content/iks-rag-thesis"
env = os.environ.copy()
env["PYTHONPATH"] = REPO_PATH + ":" + env.get("PYTHONPATH", "")

streamlit_proc = subprocess.Popen(
    [
        "streamlit", "run", "app/streamlit_app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--browser.gatherUsageStats=false",
    ],
    cwd=REPO_PATH,
    env=env,
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)
print("streamlit pid:", streamlit_proc.pid)

# (c) wait until streamlit is reachable on 8501 (models load lazily on first request)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8501", timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("streamlit didn't come up — see /content/streamlit.log")
print("streamlit is up on port 8501")

# (d) start cloudflared quick-tunnel and WAIT until it actually connects.
# IMPORTANT: force --protocol http2. cloudflared defaults to QUIC (UDP 7844);
# when that UDP path is blocked/degraded (as it intermittently is on Colab), the
# tunnel still PRINTS a *.trycloudflare.com URL but never finishes connecting, so
# the URL is "not reachable". http2 runs over TCP/443 and works through that.
# We also wait for "Registered tunnel connection" before declaring the URL ready,
# so we never hand out a named-but-dead URL. (Kill any stale tunnel first.)
subprocess.run(["pkill", "-f", "cloudflared tunnel"])
time.sleep(2)
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate",
     "--protocol", "http2",
     "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url, connected = None, False
for _ in range(120):
    line = cf_proc.stdout.readline()
    if not line:
        time.sleep(1); continue
    print(line.rstrip())
    if public_url is None:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
    if "Registered tunnel connection" in line:
        connected = True
        break
assert public_url, "cloudflared didn't print a URL — re-run this cell."
if not connected:
    print("\n(!) Tunnel URL printed but no connection registered yet — wait ~30s and "
          "refresh; if it stays unreachable, re-run this cell.")

print("\n===== OPEN THIS URL =====")
print(public_url)
print("==========================\n")

## Cell 4 — (optional) tail the streamlit log

If the app errors in the browser, run this to see the server-side traceback.

In [ ]:
!tail -n 80 /content/streamlit.log